In [1]:
import xarray as xr
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from pandas.tseries.offsets import DateOffset
import gsw
from metpy.interpolate import cross_section

/home/jupyter-vincent2/.local/lib/python3.10/site-packages/pandas/core/computation/expressions.py:21: UserWarning: Pandas requires version '2.8.4' or newer of 'numexpr' (version '2.8.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
/home/jupyter-vincent2/.local/lib/python3.10/site-packages/pandas/core/arrays/masked.py:61: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.2' currently installed).
  from pandas.core import (
/usr/lib/python3/dist-packages/scipy/__init__.py:146: UserWarning: A NumPy version >=1.17.3 and <1.25.0 is required for this version of SciPy (detected version 1.26.4
  warnings.warn(f"A NumPy version >={np_minversion} and <{np_maxversion}"


In [12]:
### Bathymetry
ncfile_etopo1 = "/media/disk2/vincent/process_profiles/data/external_supports/GEBCO_ker_large.nc"
ds_topo = xr.open_dataset(ncfile_etopo1)
elevation = ds_topo.elevation[::10,::10]

### CMA
ds = xr.open_dataset("/home/jupyter-vincent2/vincent/process_profiles/data/All/all_section_CMA.nc")

### GLORYS
ds_G = xr.open_dataset("/home/jupyter-vincent2/vincent/process_profiles/data/GLORYS/GLORYS_1000m_section_plot.nc")
### Make sure we select the same latitude and longitude between obs and model
ds_G = ds_G.sel(latitude=slice(ds.LATITUDE.min()-0.5, ds.LATITUDE.max()+0.5), longitude=slice(ds.LONGITUDE.min()-0.5, ds.LONGITUDE.max()+0.5))
ds_G = ds_G.isel(time=slice(0,len(ds_G.time)-1))

ds_G_mld = xr.open_dataset("/home/jupyter-vincent2/vincent/process_profiles/data/GLORYS/GLORYS_2026_with_mld.nc")
ds_G_mld = ds_G_mld.sel(time=slice(ds_G.time.min(), ds_G.time.max()))
ds_G = ds_G.assign(mld=ds_G_mld.mld)

In [14]:
ds_coef = ds_G.mean("time").metpy.parse_cf().squeeze()
data_c = ds.metpy.parse_cf().squeeze()

: 

In [6]:
### Cross section
start_c1 =(72,-52.5)
end_c1 = (79,-47)
cross_c1 = cross_section(data_c, start_c1[::-1], end_c1[::-1])
cross_coef1 = cross_section(ds_coef.mld, start_c1[::-1], end_c1[::-1])
cross_xiphi1 = cross_section(ds_coef.xi_phi1, start_c1[::-1], end_c1[::-1])

### Elevation cross section
cross_el1 = cross_section(-elevation.to_dataset().metpy.parse_cf(),start_c1[::-1], end_c1[::-1])

def zero_crossings_1d(da, x_coord="long", atol=1e-12):
    """
    Return zero-crossing x locations for a 1D DataArray using linear interpolation.
    Also includes exact zeros (within atol).
    """
    x = da[x_coord].values
    y = da.values

    # Keep only finite pairs
    m = np.isfinite(x) & np.isfinite(y)
    x = x[m]
    y = y[m]

    # Sign changes between consecutive points
    idx = np.where(np.signbit(y[:-1]) != np.signbit(y[1:]))[0]

    # Linear interpolation for crossing location
    den = y[idx + 1] - y[idx]
    ok = ~np.isclose(den, 0.0, atol=atol)
    x_interp = x[idx[ok]] - y[idx[ok]] * (x[idx[ok] + 1] - x[idx[ok]]) / den[ok]

    # Exact zeros
    x_exact = x[np.isclose(y, 0.0, atol=atol)]

    # Combined unique values
    return np.unique(np.concatenate([x_interp, x_exact]))


x_zero_xiphi1 = zero_crossings_1d(cross_xiphi1, x_coord="long")

if x_zero_xiphi1.size == 0:
    print("No zero crossing found for cross_xiphi1.")
else:
    print("cross_xiphi1 zero-crossing longitude(s):", x_zero_xiphi1)


AttributeError: 'Dataset' object has no attribute 'mld'

In [4]:
### Recalculating the MLD for GLORYS

def compute_mld_ds_G(ds, z_ref=10.0, density_threshold=0.03, temp_threshold=0.2):
    """
    Compute MLD for GLORYS-like dataset (ds_G) with variables:
      - so(time, depth, latitude, longitude)
      - thetao(time, depth, latitude, longitude)
      - depth(depth)

    Priority:
      1) density criterion (if so and thetao available)
      2) temperature criterion fallback
    """
    depth = ds["depth"]

    # --- Density criterion ---
    use_density = ("so" in ds) and ("thetao" in ds) and (not ds["so"].isnull().all())

    if use_density:
        # Approximate sigma0 from so/thetao (as in your workflow)
        sigma0 = xr.apply_ufunc(
            gsw.sigma0,
            ds["so"],
            ds["thetao"],
            dask="parallelized",
            output_dtypes=[float],
        )

        sigma0_ref = sigma0.interp(depth=z_ref)
        delta_sigma0 = sigma0 - sigma0_ref

        hit = delta_sigma0 >= density_threshold
        k = hit.argmax("depth")
        has_mld = hit.any("depth")

        mld = xr.where(has_mld, depth.isel(depth=k), np.nan)
        mld.name = "mld_density"
        mld.attrs["MLD_calculation"] = "density threshold from so/thetao"

    else:
        # --- Temperature fallback ---
        temp = ds["thetao"]
        temp_ref = temp.interp(depth=z_ref)
        delta_temp = abs(temp - temp_ref)

        hit = delta_temp >= temp_threshold
        k = hit.argmax("depth")
        has_mld = hit.any("depth")

        mld = xr.where(has_mld, depth.isel(depth=k), np.nan)
        mld.name = "mld_temp"
        mld.attrs["MLD_calculation"] = "temperature threshold (fallback)"

    # Remove values at max depth
    mld = mld.where(mld < depth.max())
    mld.attrs["standard_name"] = "ocean_mixed_layer_thickness"
    mld.attrs["units"] = depth.attrs.get("units", "m")
    ds["mld"] = mld
    return ds

In [5]:
ds_G = compute_mld_ds_G(ds_G)

: 

In [ ]:
ds_G.to_netcdf("/home/jupyter-vincent2/vincent/process_profiles/data/GLORYS/GLORYS_1000m_section_plot_mld.nc")

In [3]:
###Binning 
df = ds.to_dataframe().reset_index() 
df_G = ds_G.to_dataframe().reset_index()

def make_ds_cut(df, nb_bins=39, freq="M"):
    # detect coordinate column names
    lon_col = "longitude"
    lat_col = "latitude"
    time_col = "time"

    bins_dt = pd.date_range(
        start=df[time_col].min() + DateOffset(months=-1),
        end=df[time_col].max() + DateOffset(months=+1),
        freq=freq
    )

    cut_lat_label = pd.cut(df[lat_col], nb_bins)
    cut_lon_label = pd.cut(df[lon_col], nb_bins)
    cut_time_label = pd.cut(df[time_col], bins=bins_dt)

    df_cut_label = df.drop([lat_col, lon_col, time_col], axis=1)
    df_cut_label = df_cut_label.groupby([cut_time_label, cut_lon_label, cut_lat_label]).mean()

    lat_mid = pd.IntervalIndex(df_cut_label.index.get_level_values(2)).mid.unique() 
    lon_mid = pd.IntervalIndex(df_cut_label.index.get_level_values(1)).mid.unique()
    time_mid = pd.IntervalIndex(df_cut_label.index.get_level_values(0)).mid.unique()

    df_cut_label.index = df_cut_label.index.set_levels(time_mid.values, level=0)
    df_cut_label.index = df_cut_label.index.set_levels(lon_mid, level=1)
    df_cut_label.index = df_cut_label.index.set_levels(lat_mid.values, level=2)

    df_cut = df_cut_label.copy()
    df_cut.replace(0, np.nan, inplace=True)

    ds_cut = df_cut.to_xarray()
    ds_cut["latitude"] = sorted(lat_mid)
    ds_cut["longitude"] = sorted(lon_mid)
    ds_cut["time"] = time_mid

    return ds_cut, df_cut

### Apply the function and interp accoridng to ds so we have the exact same grid
ds, df = make_ds_cut(df, nb_bins=39)
ds_G, df_G = make_ds_cut(df_G, nb_bins=39)
### From here we interp ds_G to have same lat/lon grid as ds
ds_G = ds_G.interp(latitude=ds.latitude, longitude=ds.longitude, time=ds.time, method="nearest")
ds_G = ds_G.sel(time=slice(ds.time.min(), ds.time.max()))

: 

In [5]:
ds_G.to_netcdf("/home/jupyter-vincent2/vincent/process_profiles/data/processed_2026/GLORYS_gridded.nc")

In [3]:
ds_G = xr.open_dataset("/home/jupyter-vincent2/vincent/process_profiles/data/GLORYS/GLORYS.nc")
ds_G = ds_G.where(ds_G.time.dt.year < 2024,drop=True)
df_G = ds_G.to_dataframe().reset_index()
### We modify the dataset for the R analysis 
df_G['year'] = df_G.time.dt.year
df_G["mth"] = df_G.time.dt.month
df_G = df_G.drop(columns="time")
df_G = df_G.rename(columns={"latitude" : "lat","longitude" : "long"})
df_G = df_G.fillna("NA")
df_G = df_G[["long","lat","mth","year","mld"]]
df_G = df_G.sort_values(["year","mth","long","lat"])
df_G = df_G.reset_index(drop=True)

In [4]:
### Number of bins for each month = nb_lat_one_month * nb_long_one_month 
nb_lat_one_month=len(np.unique(df_G.lat))
nb_long_one_month=len(np.unique(df_G.long))
nb_bins_one_month = nb_lat_one_month * nb_long_one_month
switch = nb_long_one_month*nb_lat_one_month
switch
df_append = df_G[:switch].copy()
new_df = df_G.copy()
new_df

# ### Before this cell look at which month are missing and for which year
# year = 2024
# months = [2,3,4,5,6,7,8,9,10,11,12]
# for i in months: #months that are missing in our df_G (need to fill them for the R analysis)
#     df_append = df_G[:switch].copy()
#     df_append["mth"] = i
#     df_append["year"] = year
#     df_append = df_append.reset_index(drop=True).set_index(np.arange(df_G.index.max()+1,df_G.index.max()+(switch+1),1))
#     new_df = pd.concat((new_df,df_append))
#     new_df = new_df.sort_values(["year","mth","long","lat"])
#     new_df.reset_index(drop=True)

# new_df

,long,lat,mth,year,mld
0,60.2465,-59.7525,1,2007,34.434151
1,60.2465,-59.2300,1,2007,37.670525
2,60.2465,-58.7175,1,2007,43.022011
3,60.2465,-58.2045,1,2007,50.57011
4,60.2465,-57.6915,1,2007,57.438122
...,...,...,...,...,...
310279,79.7435,-42.3080,12,2023,23.43203
310280,79.7435,-41.7955,12,2023,31.143875
310281,79.7435,-41.2825,12,2023,31.583054
310282,79.7435,-40.7695,12,2023,25.042044


In [5]:
new_df.to_csv("/home/jupyter-vincent2/vincent/process_profiles/data/R_analysis_2026/raw/GLORYS.txt",sep=" ",index=False)